In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install --upgrade transformers accelerate
!pip install timm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 119.1 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
# KODUN EN BAŞINDAKİ İTHALAT (IMPORT) KISMINI BUNA ÇEVİR:
import os
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset
# KODUN EN BAŞINDAKİ İTHALAT (IMPORT) KISMINI BUNA ÇEVİR:
from transformers import (
    Mask2FormerImageProcessor,
    Mask2FormerForUniversalSegmentation, # 'Semantic' yerine 'Universal' kullanıyoruz
    TrainingArguments,
    Trainer
)
# ==========================================
# 1. ENVIRONMENT & PATH SETUP
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Klasör yollarını SegFormer ile aynı olacak şekilde tanımla
IMAGE_DIR = "/content/drive/MyDrive/TrainingData/Imgdir"
MASK_DIR = "/content/drive/MyDrive/TrainingData/MaskDir"
OUTPUT_DIR = "/content/drive/MyDrive/mask2former-paintings-v3-ext"

Using device: cuda


In [ ]:
# 2. RGB COLOR PALETTE & MAPPING (15 Classes)
# ==========================================
COLOR_MAP = {
    (0, 0, 0): 0,        # background
    (128, 0, 0): 1,      # building
    (0, 128, 0): 2,      # earth
    (128, 128, 0): 3,    # grass
    (0, 0, 128): 4,      # animal
    (128, 0, 128): 5,    # mountain
    (0, 128, 128): 6,    # path_road
    (128, 128, 128): 7,  # person
    (64, 0, 0): 8,       # rock
    (192, 0, 0): 9,      # shrub_bush
    (64, 128, 0): 10,    # sky
    (192, 128, 0): 11,   # tree_conical
    (64, 0, 128): 12,    # tree_broadleaf
    (192, 0, 128): 13,   # wooded_mass
    (64, 128, 128): 14   # Water
}

class_names = [
    "background", "building", "earth", "grass", "animal",
    "mountain", "path_road", "person", "rock", "shrub_bush",
    "sky", "tree_conical", "tree_broadleaf", "wooded_mass", "Water"
]

id2label = {i: name for i, name in enumerate(class_names)}
label2id = {name: i for i, name in enumerate(class_names)}
num_labels = len(class_names)

def rgb_to_id(mask_pil):
    """Renkli maskeyi Mask2Former için ID matrisine çevirir"""
    mask_np = np.array(mask_pil.convert("RGB"))
    h, w, _ = mask_np.shape
    id_mask = np.zeros((h, w), dtype=np.int64)

    for rgb, idx in COLOR_MAP.items():
        if idx == 0: continue
        match = (mask_np[:, :, 0] == rgb[0]) & (mask_np[:, :, 1] == rgb[1]) & (mask_np[:, :, 2] == rgb[2])
        id_mask[match] = idx

    return id_mask  # Mask2Former işlemcisi doğrudan numpy array kabul eder

In [ ]:
# ==========================================
# 3. MASK2FORMER DATASET CLASS (DİNAMİK ETİKET DÜZELTMESİ)
# ==========================================
class PaintingMask2FormerDataset(Dataset):
    def __init__(self, image_dir, mask_dir, processor):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.processor = processor

        self.img_files = sorted([f for f in os.listdir(image_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        self.mask_files = sorted([f for f in os.listdir(mask_dir) if f.lower().endswith('.png')])

        assert len(self.img_files) == len(self.mask_files), "Görüntü ve Maske sayıları uyuşmuyor!"

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.img_files[idx])
        mask_path = os.path.join(self.mask_dir, self.mask_files[idx])

        image = Image.open(img_path).convert("RGB")
        color_mask = Image.open(mask_path)

        id_mask_np = rgb_to_id(color_mask)

        # KRİTİK ADIM: O anki maskede gerçekten bulunan benzersiz sınıf ID'lerini buluyoruz
        # Mask2Former maliyet matrisini kurarken sadece bu mevcut sınıfları bekler
        present_classes = np.unique(id_mask_np)
        present_classes = sorted(list(present_classes)) # Küçükten büyüğe sırala

        encoded_inputs = self.processor(
            images=image,
            segmentation_maps=id_mask_np,
            return_tensors="pt"
        )

        # Güvenli Squeeze döngüsü
        for k, v in encoded_inputs.items():
            if torch.is_tensor(v):
                encoded_inputs[k] = v.squeeze(0)
            elif isinstance(v, list) and len(v) == 1:
                encoded_inputs[k] = v[0]

        # Sabit 15 sınıf yerine, sadece O RESİMDE VAR OLAN sınıfları tensor yapıp gönderiyoruz
        encoded_inputs["class_labels"] = torch.tensor(present_classes, dtype=torch.long)

        return encoded_inputs

In [ ]:
# 4. INITIALIZE PROCESSOR & MODEL
# ==========================================
model_checkpoint = "facebook/mask2former-swin-tiny-ade-semantic"

processor = Mask2FormerImageProcessor.from_pretrained(model_checkpoint)

def collate_fn(batch):
    pixel_values = torch.stack([example["pixel_values"] for example in batch])
    pixel_mask = torch.stack([example["pixel_mask"] for example in batch])
    class_labels = [example["class_labels"] for example in batch]
    mask_labels = [example["mask_labels"] for example in batch]

    return {
        "pixel_values": pixel_values,
        "pixel_mask": pixel_mask,
        "class_labels": class_labels,
        "mask_labels": mask_labels
    }

model = Mask2FormerForUniversalSegmentation.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
).to(device)

train_dataset = PaintingMask2FormerDataset(IMAGE_DIR, MASK_DIR, processor)

preprocessor_config.json:   0%|          | 0.00/538 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/82.5k [00:00<?, ?B/s]

[transformers] You passed `num_labels=15` which is incompatible to the `id2label` map of length `150`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  190MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/554 [00:00<?, ?it/s]

[transformers] Mask2FormerForUniversalSegmentation LOAD REPORT from: facebook/mask2former-swin-tiny-ade-semantic
Key                                                                                                                                | Status     |                                                                                          
-----------------------------------------------------------------------------------------------------------------------------------+------------+------------------------------------------------------------------------------------------
model.pixel_level_module.encoder.swin.encoder.layers.{0, 1, 2, 3}.blocks.{0, 1, 2, 3, 4, 5}.attention.self.relative_position_index | UNEXPECTED |                                                                                          
model.pixel_level_module.encoder.swin.layernorm.bias                                                                               | MISSING    |                                  

model.safetensors: reconstructing file:   0%|          |  0.00B /  190MB            

model.safetensors: downloading bytes:           |  0.00B            

In [ ]:
# 5. TRAINING ARGUMENTS & EXECUTION (Duyuru Hatası Düzeltildi)
# ==========================================
import os
# Deprecation uyarısını çözmek için Tensorboard dizinini çevre değişkeni olarak atıyoruz
os.environ["TENSORBOARD_LOGGING_DIR"] = f"{OUTPUT_DIR}/runs"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=6e-5,
    num_train_epochs=100,
    per_device_train_batch_size=2,
    save_strategy="epoch",
    logging_steps=5,
    remove_unused_columns=False,
    fp16=True,
    report_to="tensorboard"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate_fn
)

print("Starting Mask2Former training loop (Fixed Dataset & Logging Pipeline)...")
trainer.train()

# Model ve işlemciyi kaydet
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"[SUCCESS] Mask2Former weights completely saved to: {OUTPUT_DIR}")

Starting Mask2Former training loop (Fixed Dataset & Logging Pipeline)...


Step,Training Loss
5,89.240405
10,85.660327
15,74.210779
20,68.426074
25,64.420337
30,57.423315
35,60.707654
40,56.346320
45,53.663611
50,54.925110


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[SUCCESS] Mask2Former weights completely saved to: /content/drive/MyDrive/mask2former-paintings-v3-ext


In [ ]:
import os
import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt
from transformers import Mask2FormerImageProcessor, Mask2FormerForUniversalSegmentation

# ==========================================
# 1. AYARLAR VE PATH TANIMLAMALARI
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Eğittiğin Mask2Former modelinin kaydedildiği Drive klasörü
MODEL_DIR = "/content/drive/MyDrive/mask2former-paintings-v3"
# Deneme yapacağın yeni resimlerin klasörü
TEST_IMAGE_DIR = "/content/drive/MyDrive/SemanticSegmentationV1/ProjectImages"
# Mask2Former sonuçlarının kaydedileceği yeni klasör
OUTPUT_RESULTS_DIR = "/content/drive/MyDrive/SemanticSegmentationV1/Mask2Former_Results"

os.makedirs(OUTPUT_RESULTS_DIR, exist_ok=True)

# ==========================================
# 2. MODEL VE İŞLEMCİYİ YÜKLE
# ==========================================
print("Güncel Mask2Former v3 modeli Drive'dan yükleniyor...")
processor = Mask2FormerImageProcessor.from_pretrained(MODEL_DIR)
model = Mask2FormerForUniversalSegmentation.from_pretrained(MODEL_DIR).to(device)
model.eval()

# Sınıf isimlerini doğrudan model konfigürasyonundan çekiyoruz
class_names = model.config.id2label

# Görselleştirme için senin Pascal VOC RGB renk paletin
PALETTE = np.array([
    [0, 0, 0],        # background
    [128, 0, 0],      # building
    [0, 128, 0],      # earth
    [128, 128, 0],    # grass
    [0, 0, 128],      # animal
    [128, 0, 128],    # mountain
    [0, 128, 128],    # path_road
    [128, 128, 128],  # person
    [64, 0, 0],       # rock
    [192, 0, 0],      # shrub_bush
    [64, 128, 0],     # sky
    [192, 128, 0],    # tree_conical
    [64, 0, 128],     # tree_broadleaf
    [192, 0, 128],    # wooded_mass
    [64, 128, 128]    # Water
])

# ==========================================
# 3. TEST RESİMLERİNİ İŞLEME VE YÜZDE HESAPLAMA
# ==========================================
test_images = sorted([f for f in os.listdir(TEST_IMAGE_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

print(f"\n{len(test_images)} adet resim üzerinde Mask2Former denemesi başlatılıyor...\n")

# Create a list to accumulate raw dictionary rows
all_paintings_raw_data = []

for img_name in test_images:
    img_path = os.path.join(TEST_IMAGE_DIR, img_name)
    image = Image.open(img_path).convert("RGB")

    # Resmi model formatına getir
    inputs = processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    # KRİTİK ADIM: Mask2Former çıktılarını (logits ve maskeleri) anlamlı piksellere dönüştürüyoruz
    # post_process_semantic_segmentation fonksiyonu çıktıyı otomatik olarak orijinal resim boyutuna getirir
    predicted_semantic_maps = processor.post_process_semantic_segmentation(
        outputs,
        target_sizes=[image.size[::-1]] # (Height, Width)
    )
    pred_seg = predicted_semantic_maps[0].cpu().numpy()

    # ------------------------------------------
    # YÜZDE HESAPLAMA KISMI
    # ------------------------------------------
    total_pixels = pred_seg.size
    unique_ids, counts = np.unique(pred_seg, return_counts=True)
    pixel_counts = dict(zip(unique_ids, counts))

    print("-" * 50)
    print(f"MASK2FORMER RESİM TAHMİNİ: {img_name}")
    print("-" * 50)

   # Place this right inside the image loop, right before the class loop
    current_image_pixels = {"Image_Name": img_name}

    for class_id in range(len(class_names)):
        count = int(pixel_counts.get(class_id, 0))
        percentage = (count / total_pixels) * 100
        class_name = class_names[class_id]

        # Record the raw count into our dictionary
        current_image_pixels[class_name] = count

        if percentage > 0.0:
            print(f"  * {class_name:<15}: %{percentage:.2f} ({count} piksel)")

    # Append this image's dictionary row to our master list
    all_paintings_raw_data.append(current_image_pixels)

    # ------------------------------------------
    # MASKELİ GÖRSELİ OLUŞTURMA VE KAYDETME
    # ------------------------------------------
    # ID matrisini renkli RGB maskeye dönüştür
    color_seg = np.zeros((pred_seg.shape[0], pred_seg.shape[1], 3), dtype=np.uint8)
    for label, color in enumerate(PALETTE):
        color_seg[pred_seg == label] = color

    # Orijinal resim ile maskeyi %50 şeffaflıkla harmanla
    img_np = np.array(image)
    overlay = (img_np * 0.5 + color_seg * 0.5).astype(np.uint8)

    # Yan yana görselleştirme (Sol: Orijinal, Sağ: Mask2Former Maskesi)
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(image)
    axes[0].set_title("Orijinal Resim")
    axes[0].axis("off")

    axes[1].imshow(overlay)
    axes[1].set_title("Mask2Former Tahmini (Overlay)")
    axes[1].axis("off")

    # Görseli yeni klasöre kaydet
    save_path = os.path.join(OUTPUT_RESULTS_DIR, f"mask2former_res_{img_name}")
    plt.savefig(save_path, bbox_inches='tight')
    plt.close()

print(f"\n[BAŞARI] Mask2Former maskeli çıktıları şu klasöre kaydedildi: {OUTPUT_RESULTS_DIR}")

# ==============================================================================
# SHORT CODE ADDITION: SAVE RAW DATA TO EXCEL
# ==============================================================================
import pandas as pd

print("\nConverting raw data to Excel format...")
df_raw = pd.DataFrame(all_paintings_raw_data)

# Reorder columns nicely so Image_Name comes first, then the classes in order
ordered_cols = ["Image_Name"] + [class_names[i] for i in range(len(class_names)) if class_names[i] in df_raw.columns]
df_raw = df_raw[ordered_cols].fillna(0)

# Save file directly into your output results directory
output_excel_path = os.path.join(OUTPUT_RESULTS_DIR, "mask2former_raw_pixel_counts.xlsx")
df_raw.to_excel(output_excel_path, index=False)

print(f"👉 [RAW DATA SAVED] Created: {output_excel_path}")

Güncel Mask2Former v3 modeli Drive'dan yükleniyor...


/usr/local/lib/python3.12/dist-packages/transformers/image_processing_base.py:370: UserWarning: The following named arguments are not valid for `Mask2FormerImageProcessor.__init__` and were ignored: 'do_pad', 'reduce_labels'
  image_processor = cls(**image_processor_dict)


Loading weights:   0%|          | 0/554 [00:00<?, ?it/s]

Mask2FormerForUniversalSegmentation LOAD REPORT from: /content/drive/MyDrive/mask2former-paintings-v3
Key                                                                                                                           | Status     | 
------------------------------------------------------------------------------------------------------------------------------+------------+-
model.pixel_level_module.encoder.swin.layernorm.bias                                                                          | UNEXPECTED | 
model.pixel_level_module.encoder.swin.layernorm.weight                                                                        | UNEXPECTED | 
model.pixel_level_module.encoder.encoder.layers.{0, 1, 2, 3}.blocks.{0, 1, 2, 3, 4, 5}.attention.self.relative_position_index | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from


20 adet resim üzerinde Mask2Former denemesi başlatılıyor...

--------------------------------------------------
MASK2FORMER RESİM TAHMİNİ: N-0109-00-000032-wpu.jpg
--------------------------------------------------
  * background     : %20.06 (106557 piksel)
  * earth          : %0.24 (1298 piksel)
  * grass          : %6.33 (33644 piksel)
  * animal         : %4.44 (23580 piksel)
  * person         : %0.02 (91 piksel)
  * shrub_bush     : %5.67 (30105 piksel)
  * sky            : %10.80 (57349 piksel)
  * tree_broadleaf : %49.01 (260355 piksel)
  * wooded_mass    : %1.46 (7777 piksel)
  * Water          : %1.97 (10444 piksel)
--------------------------------------------------
MASK2FORMER RESİM TAHMİNİ: N-0134-00-000011-wpu.jpg
--------------------------------------------------
  * background     : %18.89 (89907 piksel)
  * building       : %15.63 (74377 piksel)
  * earth          : %12.93 (61533 piksel)
  * animal         : %0.35 (1645 piksel)
  * shrub_bush     : %2.46 (11715 piksel

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from PIL import Image
from transformers import (
    SegformerImageProcessor,
    SegformerForSemanticSegmentation,
    Mask2FormerImageProcessor,
    Mask2FormerForUniversalSegmentation
)

# ==========================================
# 1. AYARLAR VE PATH TANIMLAMALARI
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_TYPE = "mask2former"

MODEL_DIR = "/content/drive/MyDrive/mask2former-paintings-v3-ext"
TEST_IMAGE_DIR = "/content/drive/MyDrive/TestData/TestImages"
TEST_MASK_DIR = "/content/drive/MyDrive/TestData/TestMasks"

RESULTS_DIR = os.path.join(MODEL_DIR, "Test_Results")
os.makedirs(RESULTS_DIR, exist_ok=True)

# ==========================================
# 2. YENİ RENK HARİTASI (CVAT'IN YENİ VERDİĞİ RENKLER)
# Modelin ID sıralamasına göre ayarlandı!
# ==========================================
COLOR_MAP = {
    (0, 0, 0): 0,          # background
    (110, 13, 13): 1,      # building
    (96, 66, 7): 2,        # earth
    (131, 224, 112): 3,    # grass
    (240, 120, 240): 4,    # animal
    (37, 70, 103): 5,      # mountain
    (230, 209, 168): 6,    # path_road
    (184, 61, 245): 7,     # person
    (65, 93, 125): 8,      # rock
    (48, 173, 48): 9,      # shrub_bush
    (18, 206, 242): 10,    # sky
    (13, 135, 53): 11,     # tree_conical
    (135, 246, 171): 12,   # tree_broadleaf
    (253, 164, 5): 13,     # wooded_mass
    (85, 144, 203): 14     # Water
}

class_names = [
    "background", "building", "earth", "grass", "animal",
    "mountain", "path_road", "person", "rock", "shrub_bush",
    "sky", "tree_conical", "tree_broadleaf", "wooded_mass", "Water"
]
num_classes = len(class_names)

def rgb_to_id(mask_pil):
    mask_np = np.array(mask_pil.convert("RGB"))
    h, w, _ = mask_np.shape
    id_mask = np.zeros((h, w), dtype=np.int64)
    for rgb, idx in COLOR_MAP.items():
        if idx == 0: continue
        match = (mask_np[:, :, 0] == rgb[0]) & (mask_np[:, :, 1] == rgb[1]) & (mask_np[:, :, 2] == rgb[2])
        id_mask[match] = idx
    return id_mask

# ==========================================
# 3. MODELİ YÜKLE
# ==========================================
print(f"\n[{MODEL_TYPE.upper()}] Modeli test için yükleniyor...")
if MODEL_TYPE == "segformer":
    processor = SegformerImageProcessor.from_pretrained(MODEL_DIR)
    model = SegformerForSemanticSegmentation.from_pretrained(MODEL_DIR).to(device)
else:
    processor = Mask2FormerImageProcessor.from_pretrained(MODEL_DIR)
    model = Mask2FormerForUniversalSegmentation.from_pretrained(MODEL_DIR).to(device)
model.eval()

# ==========================================
# 4. ÇIKARIM (INFERENCE) VE KARIŞIKLIK MATRİSİ
# ==========================================
img_files = sorted([f for f in os.listdir(TEST_IMAGE_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
mask_files = sorted([f for f in os.listdir(TEST_MASK_DIR) if f.lower().endswith('.png')])

assert len(img_files) == len(mask_files), "HATA: Resim ve Maske sayıları eşit değil!"

confusion_matrix = np.zeros((num_classes, num_classes), dtype=np.int64)

print(f"{len(img_files)} adet resim üzerinde piksel piksel test yapılıyor. Lütfen bekleyin...")

for img_name, mask_name in zip(img_files, mask_files):
    image = Image.open(os.path.join(TEST_IMAGE_DIR, img_name)).convert("RGB")
    gt_id_mask = rgb_to_id(Image.open(os.path.join(TEST_MASK_DIR, mask_name)))

    inputs = processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)

    if MODEL_TYPE == "segformer":
        upsampled_logits = torch.nn.functional.interpolate(
            outputs.logits, size=image.size[::-1], mode="bilinear", align_corners=False
        )
        pred_mask = upsampled_logits.argmax(dim=1)[0].cpu().numpy()
    else:
        predicted_semantic_maps = processor.post_process_semantic_segmentation(
            outputs, target_sizes=[image.size[::-1]]
        )
        pred_mask = predicted_semantic_maps[0].cpu().numpy()

    flat_gt = gt_id_mask.flatten()
    flat_pred = pred_mask.flatten()
    indices = (flat_gt >= 0) & (flat_gt < num_classes)

    confusion_matrix += np.bincount(
        num_classes * flat_gt[indices] + flat_pred[indices],
        minlength=num_classes**2
    ).reshape(num_classes, num_classes)

# ==========================================
# 5. METRİKLERİ HESAPLA VE KAYDET
# ==========================================
print("\n" + "="*60 + f"\n{MODEL_TYPE.upper()} - METRİK HESAPLAMALARI\n" + "="*60)

tp = np.diag(confusion_matrix)
fp = np.sum(confusion_matrix, axis=0) - tp
fn = np.sum(confusion_matrix, axis=1) - tp
eps = 1e-15

iou_per_class = tp / (tp + fp + fn + eps)
precision_per_class = tp / (tp + fp + eps)
recall_per_class = tp / (tp + fn + eps)

total_pixels_per_class = np.sum(confusion_matrix, axis=1)
total_valid_pixels = np.sum(total_pixels_per_class)
class_frequencies = total_pixels_per_class / (total_valid_pixels + eps)
fwiou = np.sum(class_frequencies * iou_per_class)

# Verileri CSV için listeye ekleme ve Ekrana Yazdırma
results_data = []
present_classes_iou = []

for i, name in enumerate(class_names):
    if total_pixels_per_class[i] > 0:
        iou_val = iou_per_class[i] * 100
        prec_val = precision_per_class[i] * 100
        rec_val = recall_per_class[i] * 100
        present_classes_iou.append(iou_per_class[i])

        print(f"Sınıf: {name:<15} | IoU: %{iou_val:05.2f} | P: %{prec_val:05.2f} | R: %{rec_val:05.2f}")

        results_data.append({
            "Class Name": name,
            "IoU (%)": round(iou_val, 2),
            "Precision (%)": round(prec_val, 2),
            "Recall (%)": round(rec_val, 2)
        })

miou = np.mean(present_classes_iou) * 100
mean_precision = np.mean([precision_per_class[i] for i in range(num_classes) if total_pixels_per_class[i] > 0]) * 100
mean_recall = np.mean([recall_per_class[i] for i in range(num_classes) if total_pixels_per_class[i] > 0]) * 100
fwiou_percent = fwiou * 100

print("-" * 60)
print(f"GENEL mIoU (Mean IoU)                : %{miou:.2f}")
print(f"GENEL FWIoU (Ağırlıklı IoU)          : %{fwiou_percent:.2f}")
print(f"GENEL MEAN PRECISION (Hassasiyet)    : %{mean_precision:.2f}")
print(f"GENEL MEAN RECALL (Duyarlılık)       : %{mean_recall:.2f}")
print("=" * 60)

# CSV DOSYASINA KAYDET
df = pd.DataFrame(results_data)
csv_path = os.path.join(RESULTS_DIR, f"{MODEL_TYPE}_class_metrics.csv")
df.to_csv(csv_path, index=False)

# TXT RAPOR DOSYASINA KAYDET
txt_path = os.path.join(RESULTS_DIR, f"{MODEL_TYPE}_overall_report.txt")
with open(txt_path, "w") as f:
    f.write(f"--- {MODEL_TYPE.upper()} 6-IMAGE TEST REPORT ---\n")
    f.write(f"Mean IoU (mIoU)       : {miou:.2f}%\n")
    f.write(f"Freq Weighted IoU     : {fwiou_percent:.2f}%\n")
    f.write(f"Mean Precision        : {mean_precision:.2f}%\n")
    f.write(f"Mean Recall           : {mean_recall:.2f}%\n")

print(f"\n[BAŞARILI] İşlem tamamlandı! Sonuçlar Drive'a kaydedildi.")


[MASK2FORMER] Modeli test için yükleniyor...


Loading weights:   0%|          | 0/556 [00:00<?, ?it/s]

16 adet resim üzerinde piksel piksel test yapılıyor. Lütfen bekleyin...

MASK2FORMER - METRİK HESAPLAMALARI
Sınıf: background      | IoU: %43.34 | P: %54.85 | R: %67.38
Sınıf: building        | IoU: %60.87 | P: %89.49 | R: %65.55
Sınıf: earth           | IoU: %25.81 | P: %35.43 | R: %48.75
Sınıf: grass           | IoU: %33.00 | P: %64.38 | R: %40.38
Sınıf: animal          | IoU: %64.14 | P: %71.68 | R: %85.90
Sınıf: mountain        | IoU: %62.97 | P: %90.03 | R: %67.70
Sınıf: path_road       | IoU: %02.25 | P: %57.66 | R: %02.29
Sınıf: person          | IoU: %58.76 | P: %84.66 | R: %65.76
Sınıf: rock            | IoU: %08.98 | P: %37.64 | R: %10.54
Sınıf: shrub_bush      | IoU: %20.99 | P: %25.57 | R: %53.96
Sınıf: sky             | IoU: %96.79 | P: %98.04 | R: %98.70
Sınıf: tree_conical    | IoU: %02.66 | P: %69.24 | R: %02.69
Sınıf: tree_broadleaf  | IoU: %50.18 | P: %52.02 | R: %93.43
Sınıf: wooded_mass     | IoU: %14.97 | P: %46.64 | R: %18.06
Sınıf: Water           | IoU: %65.21 |

In [ ]:
import os
import pandas as pd

# ==========================================
# 1. FILE PATH CONFIGURATION
# ==========================================
# Change these to match your actual file paths on your system or Drive
INPUT_EXCEL_PATH = "/content/drive/MyDrive/mask2former-paintings-v3/Mask2Former_Results/mask2former_raw_pixel_counts.xlsx"
OUTPUT_EXCEL_PATH = "/content/drive/MyDrive/mask2former-paintings-v3/Mask2Former_Results/relative_ecological_cover.xlsx"
# ==========================================
# 2. DEFINE CLASS ONTOLOGY
# ==========================================
# Exact names of your 15 original model classes
ALL_CLASSES = [
    "background", "building", "earth", "grass", "animal",
    "mountain", "path_road", "person", "rock", "shrub_bush",
    "sky", "tree_conical", "tree_broadleaf", "wooded_mass", "Water"
]

# The baseline matrices we want to discard from the terrestrial space
EXCLUDE_CLASSES = ["sky", "background"]

# Define the 13 true terrestrial/ecological classes dynamically
ECOLOGICAL_CLASSES = [cls for cls in ALL_CLASSES if cls not in EXCLUDE_CLASSES]

# ==========================================
# 3. PIPELINE EXECUTION
# ==========================================
def calculate_relative_cover(input_path, output_path):
    if not os.path.exists(input_path):
        raise FileNotFoundError(f"Error: The input file '{input_path}' was not found. Please check the path.")

    print("Loading absolute metrics sheet...")
    df = pd.read_excel(input_path)

    # Verify that all required columns exist in the provided spreadsheet
    missing_cols = [col for col in ALL_CLASSES if col not in df.columns]
    if missing_cols:
        print(f"[WARNING] The following expected class columns were missing from the Excel sheet: {missing_cols}")
        print("The script will proceed using only the available ecological columns.")

    # Filter out columns that are actually present in the dataframe
    active_eco_classes = [col for col in ECOLOGICAL_CLASSES if col in df.columns]

    print("Calculating terrestrial baselines per painting...")
    # Step 1: Compute the terrestrial denominator (Sum of all ecological items per row)
    # axis=1 forces pandas to sum horizontally across columns for each individual painting
    terrestrial_canvas_sum = df[active_eco_classes].sum(axis=1)

    # Create a fresh copy of the DataFrame to append our new relative results cleanly
    df_relative = df.copy()

    print("Executing re-normalization loop...")
    # Step 2: Apply the relative cover formula for each active ecological class
    for cls in active_eco_classes:
        # Mathematical Formula: (Absolute Pixels or % / Total Terrestrial Sum) * 100
        relative_series = (df[cls] / terrestrial_canvas_sum) * 100

        # Handle edge cases: If a painting is literally 100% sky and background,
        # the terrestrial sum will be 0, resulting in NaN values. We fill these with 0.0%
        df_relative[f"{cls}_relative_percent"] = relative_series.fillna(0.0).round(2)

    # Optional Quality Check: Let's ensure our new relative columns sum up to exactly 100%
    relative_cols = [f"{cls}_relative_percent" for cls in active_eco_classes]
    df_relative["Total_Terrestrial_Check_%"] = df_relative[relative_cols].sum(axis=1).round(2)

    print(f"Exporting calculated data to: {output_path}")
    df_relative.to_excel(output_path, index=False)
    print("[SUCCESS] Data transformation pipeline complete.")

# Run the automation function
if __name__ == "__main__":
    calculate_relative_cover(INPUT_EXCEL_PATH, OUTPUT_EXCEL_PATH)

Loading absolute metrics sheet...
Calculating terrestrial baselines per painting...
Executing re-normalization loop...
Exporting calculated data to: /content/drive/MyDrive/mask2former-paintings-v3/Mask2Former_Results/relative_ecological_cover.xlsx
[SUCCESS] Data transformation pipeline complete.


In [ ]:
import os
import pandas as pd

# ==========================================
# 1. FILE PATH CONFIGURATION
# ==========================================
INPUT_EXCEL_PATH = "/content/drive/MyDrive/mask2former-paintings-v3/Mask2Former_Results/relative_ecological_cover.xlsx"
OUTPUT_EXCEL_PATH = "/content/drive/MyDrive/mask2former-paintings-v3/Mask2Former_Results/final_dafor_classification.xlsx"

# ==========================================
# 2. DEFINING THE EXACT DAFOR BINNING RULE
# ==========================================
def assign_dafor_letter(percentage):
    """
    Applies the custom macro-landscape DAFOR threshold rules:
    D: 50% - 100%
    A: 30% - 50%
    F: 15% - 30%
    O: 5% - 15%
    R: < 5% (but greater than 0)
    -: 0% (Absent)
    """
    # Force a conversion to float just in case an unexpected type slips through
    try:
        val = float(percentage)
    except (ValueError, TypeError):
        return "-"

    if val >= 50.0:
        return "D"
    elif val >= 30.0:
        return "A"
    elif val >= 15.0:
        return "F"
    elif val >= 5.0:
        return "O"
    elif val > 0.0:
        return "R"
    else:
        return "-"

# ==========================================
# 3. CORE AUTOMATION FUNCTION
# ==========================================
def run_dafor_pipeline(input_path, output_path):
    if not os.path.exists(input_path):
        raise FileNotFoundError(f"Source file not found at: {input_path}")

    print("Ingesting relative percentage dataset...")
    df = pd.read_excel(input_path)

    # Initialize a new dataframe to hold the clean final report
    df_dafor = pd.DataFrame()
    df_dafor["Image_Name"] = df["Image_Name"]

    # FIXED & UPGRADED: Robust text-cleaning and numeric force function
    def safe_get_col(col_name):
        full_name = f"{col_name}_relative_percent"

        # Check which column name format actually exists in your sheet
        if full_name in df.columns:
            col_data = df[full_name]
        elif col_name in df.columns:
            col_data = df[col_name]
        else:
            return 0.0

        # If pandas imported the column as text strings, scrub it clean
        if col_data.dtype == object:
            col_data = col_data.astype(str).str.replace('%', '', regex=False).str.strip()

        # Safely force strings into floats, turn unconvertible text into NaN, then 0.0
        return pd.to_numeric(col_data, errors='coerce').fillna(0.0)

    print("Executing functional grouping aggregations...")

    # --- STEP 4 MAPPINGS (PLANTS SUB-CATEGORIES) ---
    # Wooded: wooded_mass + tree_broadleaf + tree_conical
    df_dafor["Step4_wooded_pct"] = (
        safe_get_col("wooded_mass") +
        safe_get_col("tree_broadleaf") +
        safe_get_col("tree_conical")
    )

    # Non-Wooded: grass + shrub_bush
    df_dafor["Step4_non_wooded_pct"] = (
        safe_get_col("grass") +
        safe_get_col("shrub_bush")
    )

    # --- STEP 3 MAPPINGS (GENERAL CATEGORIES) ---
    # Plants: sum of Step 4 (Wooded + Non-Wooded)
    df_dafor["Step3_plants_pct"] = df_dafor["Step4_wooded_pct"] + df_dafor["Step4_non_wooded_pct"]

    # Animals (non-human): animal
    df_dafor["Step3_animals_pct"] = safe_get_col("animal")

    # Water: Water
    df_dafor["Step3_water_pct"] = safe_get_col("Water")

    # Soil: earth + rock + mountain
    df_dafor["Step3_soil_pct"] = (
        safe_get_col("earth") +
        safe_get_col("rock") +
        safe_get_col("mountain")
    )

    # Human-influence: building + path_road + person
    df_dafor["Step3_human_influence_pct"] = (
        safe_get_col("building") +
        safe_get_col("path_road") +
        safe_get_col("person")
    )

    print("Mapping calculated percentages into ordinal DAFOR classes...")

    # --- AUTOMATED DAFOR LETTER BINNING GATE ---
    # Step 3 General Mapping
    df_dafor["DAFOR_Step3_Plants"] = df_dafor["Step3_plants_pct"].apply(assign_dafor_letter)
    df_dafor["DAFOR_Step3_Animals"] = df_dafor["Step3_animals_pct"].apply(assign_dafor_letter)
    df_dafor["DAFOR_Step3_Water"] = df_dafor["Step3_water_pct"].apply(assign_dafor_letter)
    df_dafor["DAFOR_Step3_Soil"] = df_dafor["Step3_soil_pct"].apply(assign_dafor_letter)
    df_dafor["DAFOR_Step3_Human_Influence"] = df_dafor["Step3_human_influence_pct"].apply(assign_dafor_letter)

    # Step 4 Plants Mapping
    df_dafor["DAFOR_Step4_Wooded"] = df_dafor["Step4_wooded_pct"].apply(assign_dafor_letter)
    df_dafor["DAFOR_Step4_Non_Wooded"] = df_dafor["Step4_non_wooded_pct"].apply(assign_dafor_letter)

    # Clean up formatting: round calculated grouped percentages to 2 decimals
    pct_cols = [col for col in df_dafor.columns if col.endswith("_pct")]
    df_dafor[pct_cols] = df_dafor[pct_cols].round(2)

    # --- ORGANIZE COLS FOR SUPERIOR ACCESSIBILITY ---
    ordered_presentation_cols = [
        "Image_Name",
        "Step3_plants_pct", "DAFOR_Step3_Plants",
        "Step4_wooded_pct", "DAFOR_Step4_Wooded",
        "Step4_non_wooded_pct", "DAFOR_Step4_Non_Wooded",
        "Step3_animals_pct", "DAFOR_Step3_Animals",
        "Step3_water_pct", "DAFOR_Step3_Water",
        "Step3_soil_pct", "DAFOR_Step3_Soil",
        "Step3_human_influence_pct", "DAFOR_Step3_Human_Influence"
    ]
    df_dafor = df_dafor[ordered_presentation_cols]

    print(f"Exporting final analysis matrices to: {output_path}")
    df_dafor.to_excel(output_path, index=False)
    print("🚀 [SUCCESS] DAFOR automation completed successfully.")

if __name__ == "__main__":
    run_dafor_pipeline(INPUT_EXCEL_PATH, OUTPUT_EXCEL_PATH)

Ingesting relative percentage dataset...
Executing functional grouping aggregations...
Mapping calculated percentages into ordinal DAFOR classes...
Exporting final analysis matrices to: /content/drive/MyDrive/mask2former-paintings-v3/Mask2Former_Results/final_dafor_classification.xlsx
🚀 [SUCCESS] DAFOR automation completed successfully.


In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score

# ==========================================
# 1. FILE PATH AND COLUMN CONFIGURATION
# ==========================================
# Path to your 0-5 coded numeric Excel file
EXCEL_PATH = "/content/drive/MyDrive/Cohens_kappa_matrix.xlsx"

# The 7 distinct DAFOR categories we want to compute
categories = ["Plants", "Animals", "Water", "Soil", "Human_Influence", "Wooded", "Non_Wooded"]

# ==========================================
# 2. DATA INGESTION AND INTEGRITY CHECK
# ==========================================
if not os.path.exists(EXCEL_PATH):
    raise FileNotFoundError(f"Excel file not found at: {EXCEL_PATH}")

df = pd.read_excel(EXCEL_PATH)

# Empty lists to accumulate results for global pooling
results_per_category = []
all_ahmet_pooled = []
all_leticia_pooled = []
all_ai_pooled = []

print("=" * 70)
print("🚀 CATEGORY-BASED QUADRATIC WEIGHTED COHEN'S KAPPA ANALYSIS")
print("=" * 70)

# ==========================================
# 3. CATEGORY-SPECIFIC CALCULATION LOOP
# ==========================================
for cat in categories:
    # Dynamically map the column names matching your spreadsheet structure
    ahmet_col = f"Ahmet_{cat}"
    leticia_col = f"Leticia_{cat}"
    ai_col = f"AI_{cat}"

    # Check if all required columns exist to prevent runtime execution crashes
    if not (ahmet_col in df.columns and leticia_col in df.columns and ai_col in df.columns):
        print(f"[WARNING] Columns for category '{cat}' were not found in Excel, skipping.")
        continue

    # Clean missing values per vector segment and extract numpy arrays
    valid_data = df[[ahmet_col, leticia_col, ai_col]].dropna()

    y_ahmet = valid_data[ahmet_col].astype(int).values
    y_leticia = valid_data[leticia_col].astype(int).values
    y_ai = valid_data[ai_col].astype(int).values

    # Extend the global list arrays to preserve data vectors for the pooled system evaluation
    all_ahmet_pooled.extend(y_ahmet)
    all_leticia_pooled.extend(y_leticia)
    all_ai_pooled.extend(y_ai)

    # CRITICAL: weights='quadratic' is used to properly handle the ordinal nature of DAFOR scales
    kappa_human_baseline = cohen_kappa_score(y_ahmet, y_leticia, weights='quadratic')
    kappa_ahmet_ai = cohen_kappa_score(y_ahmet, y_ai, weights='quadratic')
    kappa_leticia_ai = cohen_kappa_score(y_leticia, y_ai, weights='quadratic')

    print(f"\n📊 Category: {cat.upper()}")
    print(f"  * Human Agreement Baseline (Ahmet vs Leticia) : {kappa_human_baseline:.3f}")
    print(f"  * Ahmet vs AI (Mask2Former)                   : {kappa_ahmet_ai:.3f}")
    print(f"  * Leticia vs AI (Mask2Former)                 : {kappa_leticia_ai:.3f}")

    results_per_category.append({
        "Category": cat,
        "Human_Baseline_Kappa": round(kappa_human_baseline, 3),
        "Ahmet_vs_AI_Kappa": round(kappa_ahmet_ai, 3),
        "Leticia_vs_AI_Kappa": round(kappa_leticia_ai, 3)
    })

# ==========================================
# 4. OVERALL POOLED SYSTEM SYSTEMIC AGREEMENT
# ==========================================
print("\n" + "=" * 70)
print("🏆 OVERALL (POOLED) SYSTEMIC AGREEMENT RESULTS (20 Images x 7 Categories)")
print("=" * 70)

pooled_human_baseline = cohen_kappa_score(all_ahmet_pooled, all_leticia_pooled, weights='quadratic')
pooled_ahmet_ai = cohen_kappa_score(all_ahmet_pooled, all_ai_pooled, weights='quadratic')
pooled_leticia_ai = cohen_kappa_score(all_leticia_pooled, all_ai_pooled, weights='quadratic')

print(f"🥇 Overall Human Baseline Agreement (Ahmet vs Leticia) : {pooled_human_baseline:.3f}")
print(f"🤖 Overall Ahmet vs AI Agreement                         : {pooled_ahmet_ai:.3f}")
print(f"👩‍🔬 Overall Leticia vs AI Agreement                       : {pooled_leticia_ai:.3f}")
print("=" * 70)

# Build and export the final comprehensive spreadsheet report
df_report = pd.DataFrame(results_per_category)

# Append the global pooled metrics row to the absolute bottom of the dataframe matrix
df_report.loc[len(df_report)] = ["OVERALL (POOLED)", round(pooled_human_baseline, 3), round(pooled_ahmet_ai, 3), round(pooled_leticia_ai, 3)]

report_output_path = os.path.dirname(EXCEL_PATH) + "/dafor_kappa_statistical_report.xlsx"
df_report.to_excel(report_output_path, index=False)

print(f"\n[SUCCESS] Statistical Kappa Report saved as Excel spreadsheet:\n👉 {report_output_path}")

🚀 CATEGORY-BASED QUADRATIC WEIGHTED COHEN'S KAPPA ANALYSIS

📊 Category: PLANTS
  * Human Agreement Baseline (Ahmet vs Leticia) : 0.641
  * Ahmet vs AI (Mask2Former)                   : 0.537
  * Leticia vs AI (Mask2Former)                 : 0.692

📊 Category: ANIMALS
  * Human Agreement Baseline (Ahmet vs Leticia) : 0.600
  * Ahmet vs AI (Mask2Former)                   : 0.267
  * Leticia vs AI (Mask2Former)                 : 0.386

📊 Category: WATER
  * Human Agreement Baseline (Ahmet vs Leticia) : 0.905
  * Ahmet vs AI (Mask2Former)                   : 0.675
  * Leticia vs AI (Mask2Former)                 : 0.636

📊 Category: SOIL
  * Human Agreement Baseline (Ahmet vs Leticia) : -0.029
  * Ahmet vs AI (Mask2Former)                   : -0.129
  * Leticia vs AI (Mask2Former)                 : 0.626

📊 Category: HUMAN_INFLUENCE
  * Human Agreement Baseline (Ahmet vs Leticia) : 0.559
  * Ahmet vs AI (Mask2Former)                   : 0.243
  * Leticia vs AI (Mask2Former)                 

In [ ]:
import os
import math
import numpy as np
import torch
import pandas as pd
from PIL import Image
from transformers import Mask2FormerImageProcessor, Mask2FormerForUniversalSegmentation

# ==============================================================================
# 1. PIPELINE RUNTIME SETUP
# ==============================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Pipeline running on device: {device}")

# Directory Configurations
MODEL_DIR = "/content/drive/MyDrive/mask2former-paintings-v3-ext"
DATASET_DIR = "/content/drive/MyDrive/full_dataset"
OUTPUT_EXCEL_PATH = "/content/drive/MyDrive/mask2former-ext_macroecological_dataset.xlsx"

# Initialize Model and Image Processor
print("Loading fine-tuned Mask2Former model from storage...")
processor = Mask2FormerImageProcessor.from_pretrained(MODEL_DIR)
model = Mask2FormerForUniversalSegmentation.from_pretrained(MODEL_DIR).to(device)
model.eval()

# Extract Class Ontology Map directly from model config
class_names = model.config.id2label  # Mapped from ID 0 to 14

# ==============================================================================
# 2. ALGORITHMIC RULE SETS (DAFOR & ECO-METRICS)
# ==============================================================================
def assign_numeric_dafor(percentage):
    """
    Converts continuous relative land cover percentages into
    standardized ordinal numerical integers:
    5 = Dominant (>= 50%)
    4 = Abundant (30% - 50%)
    3 = Frequent (15% - 30%)
    2 = Occasional (5% - 15%)
    1 = Rare (< 5% and > 0%)
    0 = Absent (Exactly 0%)
    """
    if percentage >= 50.0:
        return 5
    elif percentage >= 30.0:
        return 4
    elif percentage >= 15.0:
        return 3
    elif percentage >= 5.0:
        return 2
    elif percentage > 0.0:
        return 1
    else:
        return 0

# ==============================================================================
# 3. HIGH-THROUGHPUT BATCH RUNTIME
# ==============================================================================
image_extensions = ('.jpg', '.jpeg', '.png', '.tif', '.tiff')
all_images = sorted([f for f in os.listdir(DATASET_DIR) if f.lower().endswith(image_extensions)])

print(f"Found {len(all_images)} paintings in dataset directory. Starting batch execution...\n")
master_data_records = []

for idx, img_name in enumerate(all_images, 1):
    img_path = os.path.join(DATASET_DIR, img_name)

    try:
        # Load and verify structural properties of canvas
        image = Image.open(img_path).convert("RGB")
        width, height = image.size
        total_pixels = width * height

        # Neural Network Forward Pass
        inputs = processor(images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model(**inputs)

        # Post-Process Segmentation Map to match native image scale
        predicted_semantic_maps = processor.post_process_semantic_segmentation(
            outputs, target_sizes=[image.size[::-1]]
        )
        pred_seg = predicted_semantic_maps[0].cpu().numpy()

        # Extract pixel counts present across the canvas matrix
        unique_ids, counts = np.unique(pred_seg, return_counts=True)
        pixel_counts = dict(zip(unique_ids, counts))

        # ----------------------------------------------------------------------
        # A. TECHNICAL COMPOSITION CONTROLS
        # ----------------------------------------------------------------------
        bg_pixels = pixel_counts.get(0, 0)   # class 0: background
        sky_pixels = pixel_counts.get(10, 0) # class 10: sky

        abs_sky_pct = (sky_pixels / total_pixels) * 100.0
        abs_bg_pct = (bg_pixels / total_pixels) * 100.0

        # Calculate Terrestrial Denominator (Dropping sky & background noise)
        terrestrial_total_pixels = total_pixels - (sky_pixels + bg_pixels)

        # Dictionary row setup for spreadsheet export
        record = {
            "Image_Name": img_name,
            "Total_Pixels": total_pixels,
            "Absolute_Sky_Percent": round(abs_sky_pct, 2),
            "Absolute_Background_Percent": round(abs_bg_pct, 2),
            "Terrestrial_Total_Pixels": terrestrial_total_pixels
        }

        # ----------------------------------------------------------------------
        # B. RELATIVE ECOLOGICAL COVER COMPUTATION
        # ----------------------------------------------------------------------
        relative_percentages = {}
        for class_id, name in class_names.items():
            if name in ["sky", "background"]:
                continue

            raw_count = pixel_counts.get(class_id, 0)
            if terrestrial_total_pixels > 0:
                rel_pct = (raw_count / terrestrial_total_pixels) * 100.0
            else:
                rel_pct = 0.0

            record[f"Raw_Rel_Pct_{name}"] = round(rel_pct, 2)
            relative_percentages[name] = rel_pct

        # ----------------------------------------------------------------------
        # C. FUNCTIONAL MACRO-HABITAT AGGREGATIONS
        # ----------------------------------------------------------------------
        # Step 4: Plants Sub-classification Split
        wooded_pct = (
            relative_percentages.get("wooded_mass", 0.0) +
            relative_percentages.get("tree_broadleaf", 0.0) +
            relative_percentages.get("tree_conical", 0.0)
        )
        non_wooded_pct = (
            relative_percentages.get("grass", 0.0) +
            relative_percentages.get("shrub_bush", 0.0)
        )

        # Step 3: General Structural Habitats
        plants_pct = wooded_pct + non_wooded_pct
        animals_pct = relative_percentages.get("animal", 0.0)
        water_pct = relative_percentages.get("Water", 0.0)
        soil_pct = (
            relative_percentages.get("earth", 0.0) +
            relative_percentages.get("rock", 0.0) +
            relative_percentages.get("mountain", 0.0)
        )
        human_influence_pct = (
            relative_percentages.get("building", 0.0) +
            relative_percentages.get("path_road", 0.0) +
            relative_percentages.get("person", 0.0)
        )

        # Store continuous aggregated trends
        record["Continuous_Pct_Step4_Wooded"] = round(wooded_pct, 2)
        record["Continuous_Pct_Step4_Non_Wooded"] = round(non_wooded_pct, 2)
        record["Continuous_Pct_Step3_Plants"] = round(plants_pct, 2)
        record["Continuous_Pct_Step3_Animals"] = round(animals_pct, 2)
        record["Continuous_Pct_Step3_Water"] = round(water_pct, 2)
        record["Continuous_Pct_Step3_Soil"] = round(soil_pct, 2)
        record["Continuous_Pct_Step3_Human_Influence"] = round(human_influence_pct, 2)

        # Canopy Openness Ratio metric
        if non_wooded_pct > 0:
            record["Wooded_to_NonWooded_Ratio"] = round(wooded_pct / non_wooded_pct, 3)
        else:
            record["Wooded_to_NonWooded_Ratio"] = round(wooded_pct / 0.001, 3) # Avoid div by zero

        # ----------------------------------------------------------------------
        # D. LANDSCAPE ECOLOGY DIVERSITY METRICS
        # ----------------------------------------------------------------------
        # Array of the 5 macro-habitat proportions (proportions sum to 1.0)
        macro_habitats = [plants_pct, animals_pct, water_pct, soil_pct, human_influence_pct]
        macro_proportions = [v / 100.0 for v in macro_habitats]

        # Patch Richness: Number of distinct macro-habitats present
        landscape_richness = sum(1 for p in macro_proportions if p > 0.0)

        # Shannon Landscape Diversity Index (H')
        shannon_index = 0.0
        for p in macro_proportions:
            if p > 0.0:
                shannon_index += -p * math.log(p)

        record["Landscape_Patch_Richness"] = landscape_richness
        record["Landscape_Shannon_Index"] = round(shannon_index, 4)

        # ----------------------------------------------------------------------
        # E. ORDINAL NUMERIC DAFOR TRANSFORMATION (0 to 5)
        # ----------------------------------------------------------------------
        record["Numeric_DAFOR_Step3_Plants"] = assign_numeric_dafor(plants_pct)
        record["Numeric_DAFOR_Step4_Wooded"] = assign_numeric_dafor(wooded_pct)
        record["Numeric_DAFOR_Step4_Non_Wooded"] = assign_numeric_dafor(non_wooded_pct)
        record["Numeric_DAFOR_Step3_Animals"] = assign_numeric_dafor(animals_pct)
        record["Numeric_DAFOR_Step3_Water"] = assign_numeric_dafor(water_pct)
        record["Numeric_DAFOR_Step3_Soil"] = assign_numeric_dafor(soil_pct)
        record["Numeric_DAFOR_Step3_Human_Influence"] = assign_numeric_dafor(human_influence_pct)

        master_data_records.append(record)
        print(f"[{idx}/{len(all_images)}] Processed successfully: {img_name} (Terrestrial pixels: {terrestrial_total_pixels})")

    except Exception as e:
        print(f"[ERROR] Failed to compile structural matrix for file {img_name}: {str(e)}")
        continue

# ==============================================================================
# 4. SPREADSHEET MATRIX EXPORT
# ==============================================================================
print("\nConsolidating dataframe matrix rows...")
df_master = pd.DataFrame(master_data_records)

# Direct file output dump
print(f"Writing complete analytical master tracking spreadsheet to: {OUTPUT_EXCEL_PATH}")
df_master.to_excel(OUTPUT_EXCEL_PATH, index=False)
print("🏁 [SUCCESS] Full dataset batch ingestion layer is complete.")

Pipeline running on device: cuda
Loading fine-tuned Mask2Former model from storage...


Loading weights:   0%|          | 0/556 [00:00<?, ?it/s]

Found 243 paintings in dataset directory. Starting batch execution...

[1/243] Processed successfully: 1.jpg (Terrestrial pixels: 323147)
[2/243] Processed successfully: 10.jpg (Terrestrial pixels: 167469)
[3/243] Processed successfully: 100.jpg (Terrestrial pixels: 368466)
[4/243] Processed successfully: 101.jpg (Terrestrial pixels: 251025)
[5/243] Processed successfully: 102.jpg (Terrestrial pixels: 216679)
[6/243] Processed successfully: 103.jpg (Terrestrial pixels: 250225)
[7/243] Processed successfully: 104.jpg (Terrestrial pixels: 258585)
[8/243] Processed successfully: 105.jpg (Terrestrial pixels: 321105)
[9/243] Processed successfully: 106.jpg (Terrestrial pixels: 207993)
[10/243] Processed successfully: 107.jpg (Terrestrial pixels: 366735)
[11/243] Processed successfully: 108.jpg (Terrestrial pixels: 274681)
[12/243] Processed successfully: 109.jpg (Terrestrial pixels: 174417)
[13/243] Processed successfully: 11.jpg (Terrestrial pixels: 227555)
[14/243] Processed successfully: